# LeetCode 1999 — Smallest Greater Multiple Made of Two Digits bằng Python

Link đề: https://leetcode.com/problems/smallest-greater-multiple-made-of-two-digits/ · Độ khó: **Medium** · Chủ đề: Math, Enumeration, Breadth-First Search

> **Bài Premium:** đề trong notebook được viết lại theo hiểu biết về bài (nguồn doocs). Nếu có tài khoản Premium, nên đối chiếu lại câu chữ và ví dụ với đề gốc.

Skeleton LeetCode:

```python
class Solution:
    def findInteger(self, k: int, digit1: int, digit2: int) -> int:
        ...
```

## 1. Mục tiêu học

1. Nhận ra rằng **không gian số "chỉ gồm hai chữ số"** nhỏ hơn rất nhiều so với không gian "bội của `k`" (chỉ ~`2¹⁰` số mỗi độ dài), nên nên **liệt kê theo chữ số** thay vì theo bội.
2. Dùng `itertools.product` để sinh mọi chuỗi chữ số theo độ dài, và xử lý **số 0 đứng đầu**.
3. Cài **BFS theo chữ số** sao cho thứ tự lấy ra khỏi hàng đợi là **tăng dần theo giá trị** (ngắn trước, cùng độ dài thì nhỏ trước) — mấu chốt là **sắp xếp hai chữ số**.
4. Xử lý đúng các ràng buộc biên: **lớn hơn `k`** (không phải `≥`), **giới hạn `2³¹ − 1`**, và trường hợp cả hai chữ số đều là `0`.
5. Đối chiếu các cách với vét cạn bội số có giới hạn, trên dữ liệu ngẫu nhiên seed cố định.

## 2. Đề bài & ví dụ

Cho ba số nguyên `k`, `digit1`, `digit2`. Tìm số nguyên **nhỏ nhất** thỏa đồng thời:

- **lớn hơn** `k` (nghiêm ngặt),
- là **bội** của `k`,
- **chỉ** được viết bằng các chữ số `digit1` và/hoặc `digit2`.

Nếu không tồn tại số như vậy, hoặc số đó **vượt quá** `2³¹ − 1` (giới hạn số nguyên 32-bit có dấu), trả về `-1`.

**Ví dụ 1:** `k = 2, digit1 = 0, digit2 = 2` → `20` (20 > 2, chia hết cho 2, chỉ gồm 0 và 2).

**Ví dụ 2:** `k = 3, digit1 = 4, digit2 = 2` → `24`.

**Ví dụ 3:** `k = 2, digit1 = 0, digit2 = 0` → `-1` (chỉ viết được số 0).

**Ràng buộc:** `1 ≤ k ≤ 1000`, `0 ≤ digit1, digit2 ≤ 9`.

In [1]:
import random
import time
from collections import deque
from itertools import product

LIMIT = 2**31 - 1
examples = [((2, 0, 2), 20), ((3, 4, 2), 24), ((2, 0, 0), -1)]
for (k, d1, d2), exp in examples:
    print(f"k={k}, digit1={d1}, digit2={d2} -> kỳ vọng {exp}")
print("LIMIT =", LIMIT, "có", len(str(LIMIT)), "chữ số")

k=2, digit1=0, digit2=2 -> kỳ vọng 20
k=3, digit1=4, digit2=2 -> kỳ vọng 24
k=2, digit1=0, digit2=0 -> kỳ vọng -1
LIMIT = 2147483647 có 10 chữ số


## 3. Kiến thức nền

### 3.1 Kiểm tra một số chỉ gồm các chữ số cho trước

Chuyển số thành chuỗi, rồi dùng `set(str(x)) <= allowed` (tập con).

In [2]:
allowed = {"0", "2"}
for x in [20, 202, 21, 2]:
    print(x, set(str(x)), "->", set(str(x)) <= allowed)

20 {'0', '2'} -> True
202 {'0', '2'} -> True
21 {'1', '2'} -> False
2 {'2'} -> True


### 3.2 `itertools.product` — mọi tổ hợp có lặp

`product("24", repeat=3)` sinh mọi chuỗi độ dài 3 trên bảng chữ `{2, 4}`: `2³ = 8` chuỗi, **theo thứ tự từ điển** của bảng chữ đầu vào. Nếu bảng chữ đã sắp xếp tăng, các chuỗi cùng độ dài ra theo thứ tự **giá trị tăng dần**.

In [3]:
print(["".join(t) for t in product("24", repeat=3)])
print("Số chuỗi độ dài 1..10 trên 2 chữ số:", sum(2**L for L in range(1, 11)))

['222', '224', '242', '244', '422', '424', '442', '444']
Số chuỗi độ dài 1..10 trên 2 chữ số: 2046


### 3.3 `collections.deque` — hàng đợi cho BFS

`append` thêm vào cuối, `popleft` lấy ở đầu, cả hai `O(1)`. BFS từ các "gốc" và mỗi lần nối thêm một chữ số vào cuối tạo ra cây số; duyệt theo tầng = duyệt theo **độ dài**.

In [4]:
q = deque([2, 4])
order = []
while q and len(order) < 10:
    x = q.popleft()
    order.append(x)
    q.append(x * 10 + 2)
    q.append(x * 10 + 4)
print("BFS với chữ số (2, 4):", order)

BFS với chữ số (2, 4): [2, 4, 22, 24, 42, 44, 222, 224, 242, 244]


### 3.4 Số 0 đứng đầu và số 0

Chuỗi `"02"` có giá trị `2` — chỉ là số `2` viết thừa. Khi sinh số, ta **không cho chữ đầu là `0`** để mỗi số xuất hiện đúng một lần. Ngoài ra, số `0` không bao giờ là đáp án vì phải `> k ≥ 1`.

In [5]:
print(int("02"), int("0020"), "| chuỗi bắt đầu bằng '0' bị bỏ:", [s for s in ["02", "20", "22", "00"] if s[0] != "0"])

2 20 | chuỗi bắt đầu bằng '0' bị bỏ: ['20', '22']


## 4. Ý tưởng nền tảng / trực giác

Có hai cách "đi tìm":

- **Đi theo bội:** thử `2k, 3k, 4k, ...` và kiểm chữ số. Với `k = 2`, chữ số `{1, 3}` (mọi số đều lẻ!) thì **không bao giờ** tìm được, ta phải thử tới `2³¹/2 ≈ 10⁹` bội — quá chậm.
- **Đi theo chữ số:** số `≤ 2³¹ − 1` có tối đa 10 chữ số, mỗi vị trí chỉ có 2 lựa chọn ⇒ tổng cộng chỉ `2 + 4 + ... + 1024 = 2046` chuỗi. Thử hết là xong!

Tính tay ví dụ 2 (`k = 3`, chữ số `{2, 4}`), theo thứ tự tăng dần:
- Độ dài 1: `2, 4` — đều `≤ 3` hoặc không chia hết (4 % 3 = 1).
- Độ dài 2: `22` (22 % 3 = 1), `24` (24 % 3 = 0, 24 > 3) ⇒ **24**.

In [6]:
k = 3
for L in (1, 2):
    for t in product("24", repeat=L):
        x = int("".join(t))
        print(f"{x:3d}: >k? {x > k}, chia hết? {x % k == 0}")

  2: >k? False, chia hết? False
  4: >k? True, chia hết? False
 22: >k? True, chia hết? False
 24: >k? True, chia hết? True
 42: >k? True, chia hết? True
 44: >k? True, chia hết? False


## 5. Cách 1 — Brute Force / baseline: duyệt các bội của `k`

**Ý tưởng:** `m = 2k, 3k, ...` cho tới khi `m > 2³¹ − 1`; trả về `m` đầu tiên có chữ số hợp lệ. Bội nhỏ nhất `> k` chắc chắn bắt đầu từ `2k`.

Vì có thể chạy tới `10⁹` vòng, ta thêm tham số `max_steps` để **demo an toàn**; khi vượt `max_steps` trả về `None` (nghĩa là "chưa biết").

In [7]:
def find_by_multiples(k, d1, d2, max_steps=None):
    allowed = {str(d1), str(d2)}
    m, steps = 2 * k, 0
    while m <= LIMIT:
        if set(str(m)) <= allowed:
            return m
        m += k
        steps += 1
        if max_steps is not None and steps >= max_steps:
            return None          # chưa kết luận được
    return -1

for (k, d1, d2), exp in examples:
    got = find_by_multiples(k, d1, d2, max_steps=10**5)
    print((k, d1, d2), "->", got, "| kỳ vọng", exp)

(2, 0, 2) -> 20 | kỳ vọng 20
(3, 4, 2) -> 24 | kỳ vọng 24
(2, 0, 0) -> None | kỳ vọng -1


Ví dụ 3 trả `None`: sau `10⁵` bước vẫn chưa kết luận được, vì phải đi tới tận `2³¹` mới dám nói `-1`.

**Độ phức tạp:** worst case `O(2³¹ / k)` vòng (khoảng `10⁹` với `k = 2`), mỗi vòng `O(10)` để kiểm chữ số. Space `O(1)`.

**Nút thắt:** ta duyệt **rất nhiều** bội không liên quan, trong khi số lượng số "đẹp" (chỉ hai chữ số) rất ít.

In [8]:
t = time.perf_counter()
r = find_by_multiples(2, 1, 3, max_steps=10**6)
print("k=2, chữ số {1,3}: sau 10^6 bước ->", r, f"({time.perf_counter() - t:.2f}s); cần ~10^9 bước để kết luận -1")

k=2, chữ số {1,3}: sau 10^6 bước -> None (0.20s); cần ~10^9 bước để kết luận -1


## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** đảo chiều tìm kiếm — **liệt kê các số chỉ gồm hai chữ số** (≤ 2046 số), lọc theo `x > k` và `x % k == 0`, lấy nhỏ nhất.

**Quy tắc "nếu… thì…":**
- **Nếu** hai chữ số đều là `0` **thì** chỉ có số `0` ⇒ trả `-1` ngay.
- **Nếu** sinh số theo thứ tự **tăng dần** (độ dài tăng, cùng độ dài thì chữ số nhỏ đi trước) **thì** số đầu tiên thỏa điều kiện chính là đáp án, có thể dừng sớm.
- **Nếu** số đang xét `> 2³¹ − 1` (và ta đang sinh theo thứ tự tăng) **thì** mọi số sau cũng lớn hơn ⇒ trả `-1`.

**Invariant cho BFS:** nếu hai chữ số được sắp `a ≤ b`, hàng đợi luôn chứa các số theo **thứ tự tăng dần**. Lý do: con của `x` là `10x + a ≤ 10x + b`, và nếu `x < y` (cùng độ dài) thì `10x + b < 10y + a` (vì `10(y − x) ≥ 10 > b − a`).

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Liệt kê toàn bộ bằng `itertools.product` rồi lấy min

**Giải thích từng đoạn:**
1. `digits = sorted(set(...))` dưới dạng chuỗi.
2. Với mỗi độ dài `L = 1..10`, sinh mọi tổ hợp; **bỏ** chuỗi bắt đầu bằng `'0'`.
3. Đổi ra số, kiểm `k < x ≤ LIMIT` và `x % k == 0`; giữ min.

In [9]:
def find_enumerate(k, d1, d2):
    digits = sorted({str(d1), str(d2)})
    best = -1
    for L in range(1, 11):
        for t in product(digits, repeat=L):
            if t[0] == "0":
                continue
            x = int("".join(t))
            if k < x <= LIMIT and x % k == 0 and (best == -1 or x < best):
                best = x
    return best

for (k, d1, d2), exp in examples:
    got = find_enumerate(k, d1, d2); print((k, d1, d2), "->", got); assert got == exp

(2, 0, 2) -> 20
(3, 4, 2) -> 24
(2, 0, 0) -> -1


In [10]:
class SolutionEnumerate:
    def findInteger(self, k: int, digit1: int, digit2: int) -> int:
        digits = sorted({str(digit1), str(digit2)})
        best = -1
        for L in range(1, 11):
            for t in product(digits, repeat=L):
                if t[0] == "0":
                    continue
                x = int("".join(t))
                if k < x <= 2**31 - 1 and x % k == 0 and (best == -1 or x < best):
                    best = x
        return best

print([SolutionEnumerate().findInteger(*a) for a, _ in examples])

[20, 24, -1]


**Độ phức tạp Cách 2:** tối đa `2046` chuỗi, mỗi chuỗi dựng trong `O(10)` ⇒ `O(2^D · D)` với `D = 10` chữ số, tức **hằng số** ~`2·10⁴` thao tác. Space `O(D)`.

### 7.2 Cách 3 — Sinh theo độ dài, dừng sớm ở số đầu tiên thỏa

Vì `product` trên bảng chữ đã sắp cho thứ tự tăng trong cùng độ dài, và độ dài nhỏ hơn ⇒ số nhỏ hơn, ta có thể **trả ngay** khi gặp số đầu tiên thỏa.

In [11]:
def find_enumerate_early(k, d1, d2):
    digits = sorted({str(d1), str(d2)})
    if digits == ["0"]:
        return -1
    for L in range(1, 11):
        for t in product(digits, repeat=L):
            if t[0] == "0":
                continue
            x = int("".join(t))
            if x > LIMIT:
                return -1          # mọi số sau còn lớn hơn
            if x > k and x % k == 0:
                return x
    return -1

for (k, d1, d2), exp in examples:
    assert find_enumerate_early(k, d1, d2) == exp
print([find_enumerate_early(*a) for a, _ in examples])

[20, 24, -1]


**Độ phức tạp Cách 3:** worst case vẫn ~`2046` số (khi đáp án là `-1`), nhưng thường dừng rất sớm.

### 7.3 Cách 4 — BFS theo chữ số (số nguyên, không dùng chuỗi)

**Giải thích từng đoạn:**
1. Sắp `a ≤ b`. Nếu `b == 0` ⇒ `-1`.
2. Gốc của BFS là các chữ số **khác 0** trong `{a, b}` (theo thứ tự tăng).
3. Lấy `x` ra: nếu `x > LIMIT` ⇒ `-1`; nếu `x > k` và `x % k == 0` ⇒ trả `x`.
4. Đẩy `10x + a`, rồi `10x + b` (nếu `a ≠ b`).

Theo invariant ở mục 6, thứ tự lấy ra là tăng dần, nên lần đầu thỏa là nhỏ nhất.

In [12]:
def find_bfs(k, d1, d2):
    a, b = sorted((d1, d2))
    if b == 0:
        return -1
    starts = [d for d in sorted({a, b}) if d != 0]
    q = deque(starts)
    while q:
        x = q.popleft()
        if x > LIMIT:
            return -1
        if x > k and x % k == 0:
            return x
        q.append(x * 10 + a)
        if a != b:
            q.append(x * 10 + b)
    return -1

for (k, d1, d2), exp in examples:
    assert find_bfs(k, d1, d2) == exp
print([find_bfs(*a) for a, _ in examples])

[20, 24, -1]


**Độ phức tạp Cách 4:** mỗi số được đẩy/lấy đúng một lần; số phần tử tối đa ~`2 · 2046` (tính cả những số vượt LIMIT nằm trong hàng đợi) ⇒ `O(2^D)` thời gian và bộ nhớ, với `D = 10`.

## 8. TRACE từng bước

Trace BFS cho `k = 3, digit1 = 4, digit2 = 2` (sau khi sắp: `a = 2, b = 4`):

| Bước | Lấy ra `x` | `x > k`? | `x % k` | Đẩy vào | Hàng đợi sau bước |
|---|---|---|---|---|---|
| 1 | 2 | không | 2 | 22, 24 | [4, 22, 24] |
| 2 | 4 | có | 1 | 42, 44 | [22, 24, 42, 44] |
| 3 | 22 | có | 1 | 222, 224 | [24, 42, 44, 222, 224] |
| 4 | 24 | có | **0** | — | **trả 24** |

In [13]:
def trace_bfs(k, d1, d2):
    a, b = sorted((d1, d2))
    if b == 0:
        print("cả hai chữ số là 0 -> -1"); return -1
    q = deque(d for d in sorted({a, b}) if d != 0)
    step = 0
    while q:
        x = q.popleft(); step += 1
        if x > LIMIT:
            print(f"bước {step}: x={x} > LIMIT -> -1"); return -1
        ok = x > k and x % k == 0
        print(f"bước {step}: lấy x={x}, x>k={x > k}, x%k={x % k}", "-> TÌM THẤY" if ok else "")
        if ok:
            return x
        q.append(x * 10 + a)
        if a != b:
            q.append(x * 10 + b)
        print(f"         hàng đợi = {list(q)}")
    return -1

print("Đáp án:", trace_bfs(3, 4, 2))

bước 1: lấy x=2, x>k=False, x%k=2 
         hàng đợi = [4, 22, 24]
bước 2: lấy x=4, x>k=True, x%k=1 
         hàng đợi = [22, 24, 42, 44]
bước 3: lấy x=22, x>k=True, x%k=1 
         hàng đợi = [24, 42, 44, 222, 224]
bước 4: lấy x=24, x>k=True, x%k=0 -> TÌM THẤY
Đáp án: 24


In [14]:
print("Đáp án:", trace_bfs(7, 1, 0))

bước 1: lấy x=1, x>k=False, x%k=1 
         hàng đợi = [10, 11]
bước 2: lấy x=10, x>k=True, x%k=3 
         hàng đợi = [11, 100, 101]
bước 3: lấy x=11, x>k=True, x%k=4 
         hàng đợi = [100, 101, 110, 111]
bước 4: lấy x=100, x>k=True, x%k=2 
         hàng đợi = [101, 110, 111, 1000, 1001]
bước 5: lấy x=101, x>k=True, x%k=3 
         hàng đợi = [110, 111, 1000, 1001, 1010, 1011]
bước 6: lấy x=110, x>k=True, x%k=5 
         hàng đợi = [111, 1000, 1001, 1010, 1011, 1100, 1101]
bước 7: lấy x=111, x>k=True, x%k=6 
         hàng đợi = [1000, 1001, 1010, 1011, 1100, 1101, 1110, 1111]
bước 8: lấy x=1000, x>k=True, x%k=6 
         hàng đợi = [1001, 1010, 1011, 1100, 1101, 1110, 1111, 10000, 10001]
bước 9: lấy x=1001, x>k=True, x%k=0 -> TÌM THẤY
Đáp án: 1001


## 9. Code nộp LeetCode

In [15]:
class Solution:
    def findInteger(self, k: int, digit1: int, digit2: int) -> int:
        LIMIT = 2**31 - 1
        a, b = sorted((digit1, digit2))
        if b == 0:                         # chỉ viết được số 0
            return -1
        q = deque(d for d in sorted({a, b}) if d != 0)   # không có số 0 đứng đầu
        while q:
            x = q.popleft()                # lấy ra theo thứ tự tăng dần
            if x > LIMIT:
                return -1
            if x > k and x % k == 0:
                return x
            q.append(x * 10 + a)
            if a != b:
                q.append(x * 10 + b)
        return -1

for a_, exp in examples:
    assert Solution().findInteger(*a_) == exp
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

**Đầy đủ:** mọi số dương chỉ gồm các chữ số `{a, b}` và không có số 0 đứng đầu đều được BFS sinh ra: chữ số đầu là gốc (khác 0), mỗi chữ số tiếp theo tương ứng một lần nối `10x + d`. Mỗi số sinh ra đúng một lần (đường đi từ gốc là duy nhất).

**Thứ tự tăng dần (invariant):** khẳng định "các phần tử trong hàng đợi luôn tăng dần từ đầu tới cuối". Ban đầu đúng (gốc đã sắp). Khi lấy `x` (nhỏ nhất hiện có) và đẩy `10x + a ≤ 10x + b` vào cuối: mọi phần tử `y` đang có trong hàng đợi đều có **độ dài ≤ độ dài(x) + 1**; các phần tử cùng độ dài với con của `x` đã nằm trong hàng đợi là con của những số `x' < x` nên `10x' + b < 10x + a` (vì `10(x − x') ≥ 10 > 9 ≥ b − a`). Các phần tử ngắn hơn hiển nhiên nhỏ hơn. Vậy sau khi đẩy, hàng đợi vẫn tăng.

**Dừng đúng:** do thứ tự tăng, số đầu tiên thỏa `x > k`, `x % k == 0` là nhỏ nhất. Và khi gặp `x > LIMIT`, mọi số chưa xét đều `≥ x > LIMIT` ⇒ `-1` đúng. Vòng lặp luôn dừng vì số lượng số `≤ LIMIT` là hữu hạn (≤ 2046).

### 10.1 Khi nào đáp án là `-1`? (phân tích thêm để hiểu bài)

Một số lý do "không thể" nhận ra ngay bằng **chữ số tận cùng**:
- `k` chẵn mà cả hai chữ số đều lẻ ⇒ mọi số tạo ra đều lẻ ⇒ không chia hết cho `k`.
- `k` chia hết cho 5 mà không chữ số nào là `0` hoặc `5` ⇒ không thể tận cùng bằng 0/5.
- `k` chia hết cho 10 mà không có chữ số `0` ⇒ không thể tận cùng bằng 0.
- Còn lại có thể `-1` vì đáp án **quá lớn** (vượt `2³¹ − 1`), ví dụ `k = 999`, chữ số `{1, 1}`: bội nhỏ nhất dạng `11...1` của 999 có 27 chữ số.

Ô dưới đây đếm, với mọi `k ≤ 1000` và mọi cặp chữ số, bao nhiêu trường hợp `-1` được giải thích bởi các luật tận cùng ở trên, và bao nhiêu là do "quá lớn". Đây chỉ là phân tích để hiểu, **không cần** cho lời giải.

In [16]:
def obvious_impossible(k, d1, d2):
    ds = {d1, d2}
    if ds == {0}: return True
    if k % 2 == 0 and all(d % 2 == 1 for d in ds): return True
    if k % 5 == 0 and not (ds & {0, 5}): return True
    if k % 10 == 0 and 0 not in ds: return True
    return False

t = time.perf_counter()
total_neg = by_rule = 0
for k in range(1, 1001, 7):
    for d1 in range(10):
        for d2 in range(d1, 10):
            if Solution().findInteger(k, d1, d2) == -1:
                total_neg += 1
                by_rule += obvious_impossible(k, d1, d2)
            else:
                assert not obvious_impossible(k, d1, d2)   # luật tận cùng không được "nói sai"
print(f"k = 1, 8, 15, ... (143 giá trị): {total_neg} trường hợp -1, "
      f"{by_rule} giải thích bằng chữ số tận cùng, {total_neg - by_rule} do vượt 2^31-1 "
      f"({time.perf_counter() - t:.2f}s)")
m = next(L for L in range(1, 40) if int("1" * L) % 999 == 0)
print("k=999, chữ số {1}:", Solution().findInteger(999, 1, 1), "| số 11...1 nhỏ nhất chia hết cho 999 có", m, "chữ số")

k = 1, 8, 15, ... (143 giá trị): 3685 trường hợp -1, 2168 giải thích bằng chữ số tận cùng, 1517 do vượt 2^31-1 (0.42s)
k=999, chữ số {1}: -1 | số 11...1 nhỏ nhất chia hết cho 999 có 27 chữ số


## 11. Phân tích Time / Space

Gọi `D = 10` = số chữ số tối đa của `2³¹ − 1`.

| Cách | Time | Space |
|---|---|---|
| Bội số | `O(2³¹ / k · D)` worst | `O(1)` |
| Liệt kê `product` | `O(2^D · D)` ≈ 2·10⁴ | `O(D)` |
| Liệt kê dừng sớm | `O(2^D · D)` worst, thường nhỏ | `O(D)` |
| BFS | `O(2^D)` | `O(2^D)` |

Vì `D` cố định nên các cách 2–4 thực chất là `O(1)` theo `k`. Đo thời gian với mọi cặp chữ số cho `k = 997` (trường hợp nhiều `-1`):

In [17]:
t = time.perf_counter()
cnt_neg = 0
for d1 in range(10):
    for d2 in range(10):
        cnt_neg += Solution().findInteger(997, d1, d2) == -1
print(f"100 cặp chữ số, k=997: {cnt_neg} cặp trả -1, tổng {time.perf_counter() - t:.3f}s")

100 cặp chữ số, k=997: 66 cặp trả -1, tổng 0.009s


## 12. Bộ test

- Ví dụ đề.
- Biên: `k = 1`, `k = 1000`, chữ số trùng nhau, có số 0, đáp án sát `LIMIT`.
- Đối chiếu Cách 2, 3, 4 và lớp `Solution` trên toàn bộ `k = 1..30` và một số `k` lớn với **mọi** cặp chữ số (cách 2 là liệt kê đầy đủ, độc lập với thứ tự BFS).
- Đối chiếu với vét cạn bội số (có giới hạn bước) trên dữ liệu ngẫu nhiên seed cố định.

In [18]:
for a_, exp in examples:
    for fn in (find_enumerate, find_enumerate_early, find_bfs):
        assert fn(*a_) == exp
# biên tự dựng (đối chiếu lại đề gốc) — kiểm bằng vét cạn bội số
assert Solution().findInteger(1, 1, 1) == 11            # 1 không > 1, nên 11
assert Solution().findInteger(1, 0, 1) == 10
assert Solution().findInteger(1000, 0, 1) == 10000      # bội của 1000 > 1000
assert Solution().findInteger(2, 1, 3) == -1            # chỉ có số lẻ
assert Solution().findInteger(5, 7, 9) == -1            # không tận cùng 0 hoặc 5
assert Solution().findInteger(9, 9, 9) == 99
for a_ in [(1, 1, 1), (1, 0, 1), (1000, 0, 1), (9, 9, 9)]:
    assert find_by_multiples(*a_) == Solution().findInteger(*a_)
print("Ví dụ + biên: OK")

Ví dụ + biên: OK


In [19]:
t = time.perf_counter()
n_cases = 0
for k in list(range(1, 31)) + [97, 128, 625, 999, 1000]:
    for d1 in range(10):
        for d2 in range(d1, 10):
            r = find_enumerate(k, d1, d2)
            assert find_enumerate_early(k, d1, d2) == r == find_bfs(k, d1, d2) == Solution().findInteger(k, d2, d1)
            n_cases += 1
print(f"{n_cases} bộ (k=1..30 và vài k lớn, mọi cặp chữ số): các cách khớp nhau ({time.perf_counter() - t:.1f}s)")

1925 bộ (k=1..30 và vài k lớn, mọi cặp chữ số): các cách khớp nhau (0.8s)


In [20]:
rng = random.Random(1999)
checked = undecided = 0
for _ in range(300):
    k = rng.randint(1, 1000); d1 = rng.randint(0, 9); d2 = rng.randint(0, 9)
    want = find_by_multiples(k, d1, d2, max_steps=20000)
    got = Solution().findInteger(k, d1, d2)
    if want is None:
        # vét cạn chưa kết luận: đáp án phải là -1 hoặc > 20000 bội đầu tiên
        assert got == -1 or got > (20000 + 1) * k
        undecided += 1
    else:
        assert got == want, (k, d1, d2, got, want)
        checked += 1
print(f"Khớp vét cạn bội số: {checked} bộ; {undecided} bộ vét cạn chưa kết luận nhưng nhất quán")

Khớp vét cạn bội số: 100 bộ; 200 bộ vét cạn chưa kết luận nhưng nhất quán


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Dùng `>=` thay vì `>` (trả về chính `k`)

Đề yêu cầu **lớn hơn** `k`. Nếu `k` chỉ gồm các chữ số cho phép, bản sai trả `k`.

In [21]:
def wrong_ge(k, d1, d2):
    a, b = sorted((d1, d2))
    if b == 0: return -1
    q = deque(d for d in sorted({a, b}) if d != 0)
    while q:
        x = q.popleft()
        if x > LIMIT: return -1
        if x >= k and x % k == 0:      # SAI: >=
            return x
        q.append(x * 10 + a)
        if a != b: q.append(x * 10 + b)
    return -1

print("sai:", wrong_ge(2, 0, 2), "| đúng:", Solution().findInteger(2, 0, 2))

sai: 2 | đúng: 20


In [22]:
print("sửa: dùng x > k ->", find_bfs(2, 0, 2))

sửa: dùng x > k -> 20


### Lỗi 2 — Không sắp xếp hai chữ số trong BFS

Nếu đẩy `digit1` trước `digit2` mà `digit1 > digit2`, hàng đợi **không còn tăng dần**: với `k = 3, digit1 = 4, digit2 = 2` ta gặp `42` trước `24`.

In [23]:
def wrong_unsorted(k, d1, d2):
    if d1 == 0 and d2 == 0: return -1
    q = deque(d for d in (d1, d2) if d != 0)
    while q:
        x = q.popleft()
        if x > LIMIT: return -1
        if x > k and x % k == 0: return x
        q.append(x * 10 + d1)          # SAI: không sắp xếp
        if d1 != d2: q.append(x * 10 + d2)
    return -1

print("sai:", wrong_unsorted(3, 4, 2), "| đúng:", Solution().findInteger(3, 4, 2))

sai: 42 | đúng: 24


In [24]:
print("sửa: sắp (a, b) trước khi BFS ->", find_bfs(3, 4, 2))

sửa: sắp (a, b) trước khi BFS -> 24


### Lỗi 3 — Không giới hạn `2³¹ − 1` (vòng lặp vô hạn)

Khi không có đáp án (ví dụ `k = 2`, chữ số `{1, 3}`), BFS không giới hạn sẽ chạy mãi. Ta demo bằng cách đặt "cầu chì" `max_pops` để thấy nó vẫn chưa dừng.

In [25]:
def wrong_no_limit(k, d1, d2, max_pops=10**5):
    a, b = sorted((d1, d2))
    q = deque(d for d in sorted({a, b}) if d != 0)
    pops = 0
    while q:
        x = q.popleft(); pops += 1
        if pops > max_pops:
            return f"vẫn chạy sau {max_pops} lần lấy, x đã có {len(str(x))} chữ số"
        if x > k and x % k == 0: return x
        q.append(x * 10 + a)
        if a != b: q.append(x * 10 + b)
    return -1

print("sai:", wrong_no_limit(2, 1, 3))

sai: vẫn chạy sau 100000 lần lấy, x đã có 16 chữ số


In [26]:
print("sửa (có kiểm x > LIMIT):", Solution().findInteger(2, 1, 3))

sửa (có kiểm x > LIMIT): -1


### Lỗi 4 — Cho phép số 0 làm gốc BFS

Nếu bắt đầu từ `0` thì `0 % k == 0`; bản sai quên điều kiện `x > k` sẽ trả `0`. Còn nếu có kiểm `x > k` nhưng chữ số là `{0, 0}`, hàng đợi chỉ toàn số `0` và lặp vô hạn. Dưới đây minh họa trường hợp trả `0`.

In [27]:
def wrong_zero_root(k, d1, d2):
    q = deque([0])                     # SAI: gốc 0
    while q:
        x = q.popleft()
        if x > LIMIT: return -1
        if x % k == 0 and x != k:      # SAI: thiếu x > k, 0 lọt vào
            return x
        for d in sorted({d1, d2}):
            q.append(x * 10 + d)
    return -1

print("sai:", wrong_zero_root(2, 0, 2), "| đúng:", Solution().findInteger(2, 0, 2))

sai: 0 | đúng: 20


In [28]:
print("sửa: gốc chỉ gồm chữ số khác 0 ->", find_bfs(2, 0, 2))

sửa: gốc chỉ gồm chữ số khác 0 -> 20


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Số chỉ gồm vài chữ số cho trước"** ⇒ không gian nhỏ (`c^D`), **liệt kê theo chữ số**.
- **"Nhỏ nhất thỏa điều kiện"** + sinh theo thứ tự tăng ⇒ dừng ở phần tử đầu tiên thỏa.
- BFS nối chữ số cho thứ tự tăng **chỉ khi** bảng chữ số đã sắp và không có số 0 đứng đầu.
- Luôn tự hỏi ba biên: `>` hay `≥`? giới hạn tràn số? trường hợp chỉ viết được `0`?
- Khi phải chứng minh "không tồn tại" bằng duyệt bội ⇒ dấu hiệu nên đảo chiều tìm kiếm.

## 15. Biến thể của bài

1. **Không giới hạn độ dài, tìm bội nhỏ nhất gồm chữ số {0, 1}** (bài kinh điển): BFS theo **phần dư mod k** (chỉ `k` trạng thái), trả về dạng chuỗi.
2. **Ba chữ số trở lên:** liệt kê `c^D` vẫn được với `c` nhỏ.
3. **Số lớn nhất `< X`** chỉ gồm các chữ số cho trước: duyệt theo độ dài giảm dần, hoặc tham lam từng chữ số.
4. **Đếm** số lượng bội của `k` chỉ gồm các chữ số cho trước trong `[1, N]`: digit DP với trạng thái `(vị trí, phần dư, chặn trên)`.

Minh họa biến thể 1 (BFS theo phần dư, tự dựng — kiểm bằng vét cạn):

In [29]:
def smallest_01_multiple(k):
    """Bội dương nhỏ nhất của k chỉ gồm chữ số 0/1, trả chuỗi."""
    parent = {1 % k: (None, "1")}
    q = deque([1 % k])
    while q:
        r = q.popleft()
        if r == 0:
            s = []
            while r is not None:
                p, d = parent[r]; s.append(d); r = p
            return "".join(reversed(s))
        for d in (0, 1):
            nr = (r * 10 + d) % k
            if nr not in parent:
                parent[nr] = (r, str(d)); q.append(nr)

def smallest_01_brute(k):
    # liệt kê số 0/1 theo thứ tự tăng: 1, 10, 11, 100, ... (= biểu diễn nhị phân của 1, 2, 3, ...)
    b = 1
    while int(bin(b)[2:]) % k:
        b += 1
    return bin(b)[2:]

for k in range(1, 60):
    assert smallest_01_multiple(k) == smallest_01_brute(k)
print({k: smallest_01_multiple(k) for k in (3, 7, 9, 99)})

{3: '111', 7: '1001', 9: '111111111', 99: '111111111111111111'}


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Duyệt bội | `2k, 3k, ...` kiểm chữ số | `O(2³¹/k)` | `O(1)` | chỉ khi đáp án chắc chắn nhỏ |
| 2. `product` + min | liệt kê ≤ 2046 số | `O(2^D·D)` | `O(D)` | dễ viết, dễ đúng |
| 3. `product` dừng sớm | sinh tăng dần, gặp là trả | `O(2^D·D)` worst | `O(D)` | nhanh hơn trung bình |
| 4. BFS số nguyên | hàng đợi tăng dần | `O(2^D)` | `O(2^D)` | **nộp bài**, mở rộng được |

## 17. Bài tập (từ dễ đến khó)

1. **Đoán output:** `findInteger(5, 5, 0)`.
2. **Tính tay:** `findInteger(7, 1, 0)` — liệt kê các số 0/1 theo thứ tự và tính phần dư cho 7.
3. **Điền TODO:** hoàn thiện `all_two_digit_numbers(d1, d2, max_len)` trả về **danh sách tăng dần** mọi số dương gồm chữ số `d1`/`d2`, độ dài ≤ `max_len`.
4. **Tự viết lại:** viết lại BFS mà không nhìn code.
5. **Mini challenge:** viết `count_valid(k, d1, d2)` đếm có bao nhiêu số `x` với `k < x ≤ 2³¹ − 1`, `x % k == 0`, chỉ gồm `d1`/`d2`.

In [30]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def all_two_digit_numbers(d1, d2, max_len):
    res = []
    # TODO: sinh theo độ dài, bỏ số 0 đứng đầu, trả về tăng dần
    return res

In [31]:
# Bài 4 — TODO (chỉ định nghĩa, không gọi)
def my_find_bfs(k, d1, d2):
    # TODO: sắp chữ số, xử lý {0,0}, BFS từ gốc khác 0, kiểm LIMIT
    pass

In [32]:
# Bài 5 — TODO (chỉ định nghĩa, không gọi)
def count_valid(k, d1, d2):
    # TODO
    pass

## 18. Gợi ý 3 mức

**Bài 1.** (1) Bội của 5 tận cùng bằng 0 hoặc 5. (2) Thứ tự: 5, 50, 55, ... (3) 5 không > 5; 50 chia hết cho 5.

**Bài 2.** (1) Thứ tự: 1, 10, 11, 100, 101, 110, 111, 1000, 1001. (2) Phần dư: 1, 3, 4, 2, 3, 5, 6, 6, 0. (3) Đáp án là số có phần dư 0 đầu tiên.

**Bài 3.** (1) Dùng `product(sorted({str(d1), str(d2)}), repeat=L)`. (2) Bỏ `t[0] == '0'`. (3) Dùng `set` khi `d1 == d2` để tránh trùng.

**Bài 4.** (1) `a, b = sorted(...)`. (2) Gốc `d != 0`. (3) Kiểm `x > LIMIT` trước khi kiểm chia hết.

**Bài 5.** (1) Liệt kê như bài 3 với `max_len = 10`. (2) Lọc `k < x ≤ LIMIT`. (3) Đếm `x % k == 0`.

## 19. Đáp án tham khảo

In [33]:
# Bài 1, 2
assert Solution().findInteger(5, 5, 0) == 50
assert Solution().findInteger(7, 1, 0) == 1001
print("Bài 1: 50 | Bài 2: 1001; phần dư:", [x % 7 for x in [1, 10, 11, 100, 101, 110, 111, 1000, 1001]])

Bài 1: 50 | Bài 2: 1001; phần dư: [1, 3, 4, 2, 3, 5, 6, 6, 0]


In [34]:
# Bài 3
def all_two_digit_numbers_ans(d1, d2, max_len):
    digits = sorted({str(d1), str(d2)})
    res = []
    for L in range(1, max_len + 1):
        for t in product(digits, repeat=L):
            if t[0] != "0":
                res.append(int("".join(t)))
    return res

r = all_two_digit_numbers_ans(4, 2, 3)
assert r == sorted(r) and len(r) == 14
assert all_two_digit_numbers_ans(0, 0, 5) == []
assert all_two_digit_numbers_ans(0, 3, 3) == [3, 30, 33, 300, 303, 330, 333]
print("Bài 3:", r)

Bài 3: [2, 4, 22, 24, 42, 44, 222, 224, 242, 244, 422, 424, 442, 444]


In [35]:
# Bài 4
def my_find_bfs_ans(k, d1, d2):
    a, b = sorted((d1, d2))
    if b == 0: return -1
    q = deque(d for d in sorted({a, b}) if d != 0)
    while q:
        x = q.popleft()
        if x > LIMIT: return -1
        if x > k and x % k == 0: return x
        q.append(10 * x + a)
        if a != b: q.append(10 * x + b)
    return -1

# Bài 5
def count_valid_ans(k, d1, d2):
    return sum(1 for x in all_two_digit_numbers_ans(d1, d2, 10) if k < x <= LIMIT and x % k == 0)

rng = random.Random(55)
for _ in range(150):
    k = rng.randint(1, 1000); d1 = rng.randint(0, 9); d2 = rng.randint(0, 9)
    assert my_find_bfs_ans(k, d1, d2) == find_enumerate(k, d1, d2)
    cv = count_valid_ans(k, d1, d2)
    assert (cv == 0) == (find_enumerate(k, d1, d2) == -1)
print("Bài 4: OK | Bài 5: count_valid(3, 2, 4) =", count_valid_ans(3, 2, 4))

Bài 4: OK | Bài 5: count_valid(3, 2, 4) = 340


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template BFS sinh số tăng dần từ tập chữ số:**

```python
digs = sorted(set(digits))
q = deque(d for d in digs if d != 0)
while q:
    x = q.popleft()                 # tăng dần
    if x > LIMIT: return -1
    if ok(x): return x
    for d in digs:
        q.append(10 * x + d)
```

In [36]:
def template(k, digits):
    digs = sorted(set(digits))
    q = deque(d for d in digs if d != 0)
    while q:
        x = q.popleft()
        if x > LIMIT: return -1
        if x > k and x % k == 0: return x
        for d in digs: q.append(10 * x + d)
    return -1
print([template(k, (d1, d2)) for (k, d1, d2), _ in examples])

[20, 24, -1]


**Checklist:**

- [ ] `x > k` (nghiêm ngặt), không phải `x >= k`.
- [ ] Sắp hai chữ số trước khi BFS / `product`.
- [ ] Không dùng `0` làm chữ số đầu; trường hợp `{0, 0}` trả `-1`.
- [ ] Kiểm `x > 2³¹ − 1` để dừng và trả `-1`.
- [ ] `digit1 == digit2` không đẩy trùng.
- [ ] Đã thử: `k = 1`, `k = 1000`, `(2, 1, 3)`, `(5, 7, 9)`, ví dụ đề.